In [ ]:
!pip install --upgrade transformers accelerate peft

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
# Imports
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
import numpy as np
import torch

# Dataset IMDb
dataset = load_dataset("imdb")
dataset = dataset.rename_columns({"text": "sentence", "label": "label"})
print(dataset)

# Tokenizador y modelo
model_checkpoint = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding=False, max_length=512)

tokenized_datasets = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Métricas
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"],
    }

# Cargar modelo (2 clases, ignora mismatch)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./results-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    push_to_hub=False,
    report_to="none"  # ← Add this line to disable W&B
)

# Definir Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42),
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Fine-tuning
trainer.train()

# Evaluación
results = trainer.evaluate()
print("\nResultados finales:", results)

# Guardar modelo fine-tuneado
trainer.save_model("./distilbert-imdb-finetuned")

# Prueba rápida de inferencia
texts = [
    "This movie was amazing! The performances were outstanding.",
    "It was a complete waste of time. Terrible acting and plot.",
    "Not bad, but could have been shorter."
]

# Tokeniza (crea tensores en la CPU)
inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True)

inputs = inputs.to(model.device)

outputs = model(**inputs)

preds = torch.argmax(outputs.logits, dim=1).cpu()

for t, p in zip(texts, preds):
    print(f"\n{t}\n→ Sentimiento: {'Positivo' if p == 1 else 'Negativo'}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at lxyuan/distilbert-base-multilingual-cased-sentiments-student and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1357736088.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.309500,0.239678,0.905200,0.906678
2,0.187800,0.234196,0.914280,0.914015
3,0.130800,0.331144,0.913840,0.914306



Resultados finales: {'eval_loss': 0.3311438262462616, 'eval_accuracy': 0.91384, 'eval_f1': 0.9143061744112031, 'eval_runtime': 367.0452, 'eval_samples_per_second': 68.112, 'eval_steps_per_second': 4.258, 'epoch': 3.0}

This movie was amazing! The performances were outstanding.
→ Sentimiento: Positivo

It was a complete waste of time. Terrible acting and plot.
→ Sentimiento: Negativo

Not bad, but could have been shorter.
→ Sentimiento: Positivo
